In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS my_pipeline.bronze;

CREATE SCHEMA IF NOT EXISTS my_pipeline.silver;

CREATE SCHEMA IF NOT EXISTS my_pipeline.gold;

In [0]:
%sql
SHOW SCHEMAS IN my_pipeline;

In [0]:
url ="https://archive-api.open-meteo.com/v1/archive?latitude=13.0827&longitude=80.2707&start_date=2026-07-01&end_date=2026-07-07&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code,rain,apparent_temperature,precipitation,surface_pressure,cloud_cover,wind_direction_10m,wind_gusts_10m"

In [0]:
import requests
response = requests.get(url)
data=response.json()
print(data)

In [0]:
hourly=data["hourly"]
print(hourly.keys())

In [0]:
rows = list(zip(
    hourly["time"],
    hourly["temperature_2m"],
    hourly["relative_humidity_2m"],
    hourly["apparent_temperature"],
    hourly["precipitation"],
    hourly["rain"],
    hourly["weather_code"],
    hourly["surface_pressure"],
    hourly["cloud_cover"],
    hourly["wind_speed_10m"],
    hourly["wind_direction_10m"],
    hourly["wind_gusts_10m"]
))

columns = [
    "time",
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "weather_code",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m"
]

df = spark.createDataFrame(rows, columns)

display(df)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_df = (
    df
    .withColumn("location", lit("Chennai"))
    .withColumn("latitude", lit(13.0827))
    .withColumn("longitude", lit(80.2707))
    .withColumn("source", lit("Open-Meteo Historical Weather API"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

display(bronze_df)

In [0]:
print("rows: ",bronze_df.count())
print("columns: ",len(bronze_df.columns))

In [0]:
bronze_df.write.format("delta").mode("append").saveAsTable("my_pipeline.bronze.weather")

In [0]:
#bronze_check=spark.table('my_pipeline.bronze.weather')
#print("Rows: ",bronze_check.count())
#print("Columns: ",len(bronze_check.columns))

##SILVER

In [0]:
silver_df=spark.table("my_pipeline.bronze.weather")
silver_df.display()

In [0]:
from pyspark.sql.functions import *
silver_df.select([sum(when(col(c).isNull() ,1).otherwise(0)).alias(c) for c in silver_df.columns]).show()

In [0]:
total_rows=silver_df.count()
unique_rows=silver_df.dropDuplicates().count() 
print("Total rows: ",total_rows)
print("Unique rows: ",unique_rows)


In [0]:
print("Total timestamps:", silver_df.select("time").count())
print("Unique timestamps:", silver_df.select("time").distinct().count())

In [0]:
silver_df.printSchema()

In [0]:
silver_df=silver_df.withColumn("time",to_timestamp("time"))
silver_df.printSchema()

In [0]:
silver_df.select("time").limit(10).display()

In [0]:
silver_df=silver_df.withColumn("date",to_date("time"))
silver_df.select("date").display()

In [0]:
silver_df.columns

In [0]:
silver_df=silver_df.drop("location","latitude","longitude","source","ingestion_timestamp")
silver_df.columns

In [0]:
silver_df.summary().display()

In [0]:
#dropduplicates
silver_df=silver_df.dropDuplicates()
print("rows after removig duplicates",silver_df.count())

In [0]:
#arrange cols in order
silver_df = silver_df.select(
    "time",
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "weather_code",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m"
)

display(silver_df)

In [0]:
print("Rows:", silver_df.count())
print("Columns:", len(silver_df.columns))
print(silver_df.columns)

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("my_pipeline.silver.weather")

In [0]:
silver_check=spark.table("my_pipeline.silver.weather")
print("rows: ",silver_check.count())
print("columns: ",len(silver_check.columns))

##GOLD

In [0]:
gold_df=spark.table("my_pipeline.silver.weather")
gold_df.display()

In [0]:
from pyspark.sql.functions import to_timestamp, to_date

silver_df = spark.table("my_pipeline.bronze.weather")

silver_df = (
    silver_df
    .withColumn("time", to_timestamp("time"))
    .withColumn("date", to_date("time"))
)

In [0]:
gold_df=silver_df.groupBy("date").agg(
    avg("temperature_2m").alias("avg_temperature"),
    min("temperature_2m").alias("min_temperature"),
    max("temperature_2m").alias("max_temperature"),
    avg("apparent_temperature").alias("avg_apparente_temperature"),
    avg("relative_humidity_2m").alias("avg_relative_humidity"),
    sum("precipitation").alias("total_precipation"),
    sum("rain").alias("total_rain"),
    max("wind_speed_10m").alias("max_wind_speed_10m"),
    max("wind_gusts_10m").alias("max_wind_guests_10m"),
    avg("cloud_cover").alias("avg_cloud_cover")
)

In [0]:
gold_df.columns

In [0]:
gold_df.write.format("delta").mode("overwrite").saveAsTable("my_pipeline.gold.weather")

In [0]:
gold_check = spark.table("my_pipeline.gold.weather")

print("Rows:", gold_check.count())
print("Columns:", len(gold_check.columns))

display(gold_check)